![DB Academy](./Includes/images/db-academy.png)

# Lab - Improve the Knowledge Store for the HR Genie Space

#### Duration: ~20 Minutes

## Overview

In the previous lab, you created an HR Genie Space, added tables, and established baseline benchmarks. Some of those benchmarks likely failed because Genie was working with **table and column names only** — no metadata, no SQL logic, no instructions.

In this lab, you will **iteratively improve** your HR Genie Space using the same three-phase approach from the demonstrations:

1. **Metadata** — Table and column descriptions, synonyms, and prompt matching
2. **SQL Logic** — Joins, SQL expressions, and example SQL queries
3. **Instructions** — Text instructions defining business terms and default behaviors

After each phase, you will re-run your benchmarks to measure progress. The goal is to get **all benchmarks passing**.

## Learning Objectives

By the end of this lab, you will be able to:

1. **Add table and column descriptions** to Unity Catalog tables using `ALTER TABLE` SQL statements.
2. **Add column synonyms** in the Knowledge Store to map business terms to column names.
3. **Enable prompt matching** (format assistance and entity matching) on key categorical columns.
4. **Add a join** in the Knowledge Store to define table relationships.
5. **Add a SQL expression** to create a derived field for Genie.
6. **Add example SQL queries** to teach Genie complex query patterns.
7. **Add text instructions** to define ambiguous business terms and set default behaviors.
8. **Re-run benchmarks** after each phase and measure improvement.


<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Prerequisites</strong>
  <div style="color:#333;">

Complete **Lab 05 - Create an HR Genie**. This lab builds on the HR Genie Space and benchmarks you created in Lab 05.

  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Serverless SQL Warehouse</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless SQL Warehouse (2X-Small is sufficient)**
  - Select **Compute** > **More** > **SQL Warehouse** > Select a SQL Warehouse you have access to.

**NOTE:** This notebook was **developed and tested using Serverless SQL Warehouse**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Review - Where You Left Off

In **05 Lab - Create an HR Genie Space**, you created an HR Genie Space with 4 tables and added benchmarks.

##### Without any metadata or instructions, your results likely looked like this:

<div style="margin: 16px 0;">
  <strong style="display:block; margin-bottom:12px; font-size: 14pt; color: #0b2026;">
    Expected Baseline Results (Results May Vary)
  </strong>

<style>
table td, table th {
  font-size: 14pt !important;
}
</style>

<table style="width: 100%; border-collapse: collapse; line-height: 1.5;">
  <thead>
    <tr style="background: #1B5162; color: white;">
      <th style="padding: 10px 14px; text-align: center; width: 40px; border: 1px solid #EEEDE9;">#</th>
      <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">Benchmark Question</th>
      <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">Likely Baseline Result</th>
      <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">Why</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background: #E8F7EF;">
      <td style="padding: 10px 14px; text-align: center; border: 1px solid #EEEDE9;">1</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;"><em>How many active employees do we have?</em></td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700; color: #00A972;">Passed</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Simple filter — column names are clear</td>
    </tr>
    <tr style="background: #E8F7EF;">
      <td style="padding: 10px 14px; text-align: center; border: 1px solid #EEEDE9;">2</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;"><em>What is the average salary by department?</em></td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700; color: #00A972;">Passed</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Descriptive column names make the join obvious</td>
    </tr>
    <tr style="background: #FDECEA;">
      <td style="padding: 10px 14px; text-align: center; border: 1px solid #EEEDE9;">3</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;"><em>Who are our most underpaid employees?</em></td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700; color: #98102A;">Failed</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">"Underpaid" is ambiguous — Genie doesn't know it means <code>comp_ratio &lt; 1.0</code></td>
    </tr>
    <tr style="background: #FDECEA;">
      <td style="padding: 10px 14px; text-align: center; border: 1px solid #EEEDE9;">4</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;"><em>Which managers have the highest attrition on their teams?</em></td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700; color: #98102A;">Failed</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">"Attrition" isn't in the schema + requires three-table join</td>
    </tr>
  </tbody>
</table>

</div>

Your goal in this lab is to **make all benchmarks pass** by iteratively adding context to the Knowledge Store using the three-phase approach from the demonstrations.

## B. Classroom Setup

1. Run the cell below to set your default catalog and schema.

In [0]:
%run ./Includes/Classroom-Setup-lab-setup

<div style="max-width: 1200px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 22px 26px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="./Includes/images/icons/genie-code.png" style="height: 44px; margin-bottom: 10px;">

  <div style="font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 12px;">
    Need Help? Use Genie Code
  </div>

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.6; margin-bottom: 16px;">
    Genie is an AI-powered assistant that can help you as you work through this lab.
    Use it if you get stuck or want a little extra guidance.
  </div>

  <div style="display: flex; gap: 10px;">
    <a href="https://docs.databricks.com/aws/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 22px; border-radius: 8px; text-decoration: none;">
      AWS →
    </a>
    <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-code/use-genie-code" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 22px; border-radius: 8px; text-decoration: none;">
      Azure →
    </a>
    <a href="https://docs.databricks.com/gcp/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 22px; border-radius: 8px; text-decoration: none;">
      GCP →
    </a>
  </div>

</div>

</div>


## C. Step 1: Add Table and Column Metadata

Metadata are the **easiest, highest-impact** change you can make. They tell Genie what each table contains, what each column means, and what values are valid.

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    What Good Metadata Includes
  </strong>
  <div style="color:#333;">

- **Table descriptions**: What does this table represent? When should Genie use it?
- **Column descriptions**: What does this column mean? What are the valid values? How does it join to other tables?
- **Primary/foreign key constraints**: Informational metadata that tells Genie how tables join. These are `NOT ENFORCED` — they are hints, not data-level constraints.

  </div>
</div>

### C1. Add Metadata Using `ALTER TABLE`

The SQL below adds table descriptions, column descriptions, and primary/foreign key constraints to all four HR tables at once.

1. **Review the metadata** being added as the code executes. Pay attention to:
   - How column descriptions clarify ambiguous terms (e.g., what does **comp_ratio** mean?)
   - How join relationships are made explicit through foreign key constraints
   - How valid values are listed for categorical columns like **status** and **reason_code**

2. **TO DO: Feel free to make any changes you see fit.**

3. Run the cell below.

In [0]:
-- -----------------------------------------------------------
-- employees
-- -----------------------------------------------------------
COMMENT ON TABLE employees
  IS 'Employee profiles for the company. Each row represents one unique employee. Use employee_id to join with compensation, and manager_id to join with managers. The status column indicates whether the employee is currently Active, Resigned, or Terminated.';

ALTER TABLE employees
  ALTER COLUMN employee_id COMMENT 'Unique identifier for each employee (e.g., EMP-0001). Primary key. Join key to compensation.employee_id and resignations.employee_id.';
ALTER TABLE employees
  ALTER COLUMN first_name COMMENT 'Employee first name.';
ALTER TABLE employees
  ALTER COLUMN last_name COMMENT 'Employee last name.';
ALTER TABLE employees
  ALTER COLUMN department COMMENT 'Department the employee belongs to. Valid values: Engineering, Product, Operations, Finance, Marketing, Sales, Customer Success.';
ALTER TABLE employees
  ALTER COLUMN region COMMENT 'Geographic region where the employee is based. Valid values: West Coast, Southwest, Midwest, Northeast, Southeast.';
ALTER TABLE employees
  ALTER COLUMN hire_date COMMENT 'Date the employee was hired.';
ALTER TABLE employees
  ALTER COLUMN manager_id COMMENT 'Foreign key to managers.manager_id. Identifies the employee direct manager.';
ALTER TABLE employees
  ALTER COLUMN job_title COMMENT 'Employee job title (e.g., Senior Engineer, Staff Engineer, Product Manager).';
ALTER TABLE employees
  ALTER COLUMN job_level COMMENT 'Numeric job level. Higher numbers indicate more senior roles.';
ALTER TABLE employees
  ALTER COLUMN status COMMENT 'Current employment status. Valid values: Active (currently employed), Resigned (voluntarily left), Terminated (involuntarily separated). Use this column to filter for current vs. former employees.';

ALTER TABLE employees ALTER COLUMN employee_id SET NOT NULL;
ALTER TABLE employees ADD CONSTRAINT pk_employees PRIMARY KEY (employee_id) NOT ENFORCED;

-- -----------------------------------------------------------
-- managers (must come before employees FK)
-- -----------------------------------------------------------
COMMENT ON TABLE managers
  IS 'Manager profiles. Each row represents a manager with their department, region, team size, and performance rating. Use manager_id to join with employees.manager_id. Managers with a non-null end_date are no longer active.';

ALTER TABLE managers
  ALTER COLUMN manager_id COMMENT 'Unique identifier for each manager (e.g., MGR-001). Primary key. Join key to employees.manager_id.';
ALTER TABLE managers
  ALTER COLUMN manager_name COMMENT 'Full name of the manager.';
ALTER TABLE managers
  ALTER COLUMN department COMMENT 'Department the manager leads. Valid values match employees.department.';
ALTER TABLE managers
  ALTER COLUMN region COMMENT 'Geographic region where the manager is based. Valid values match employees.region.';
ALTER TABLE managers
  ALTER COLUMN team_size COMMENT 'Number of direct reports for this manager.';
ALTER TABLE managers
  ALTER COLUMN start_date COMMENT 'Date the manager started in this role.';
ALTER TABLE managers
  ALTER COLUMN end_date COMMENT 'Date the manager left this role. NULL means the manager is currently active.';
ALTER TABLE managers
  ALTER COLUMN performance_rating COMMENT 'Manager performance rating on a 1.0 to 5.0 scale. Higher is better.';

ALTER TABLE managers ALTER COLUMN manager_id SET NOT NULL;
ALTER TABLE managers ADD CONSTRAINT pk_managers PRIMARY KEY (manager_id) NOT ENFORCED;

-- -----------------------------------------------------------
-- compensation
-- -----------------------------------------------------------
COMMENT ON TABLE compensation
  IS 'Compensation details for each employee. Each row contains salary, market benchmark, and raise history. Use employee_id to join with the employees table. The comp_ratio column indicates whether an employee is paid above (> 1.0) or below (< 1.0) market rate.';

ALTER TABLE compensation
  ALTER COLUMN employee_id COMMENT 'Foreign key to employees.employee_id. Each employee has exactly one compensation record.';
ALTER TABLE compensation
  ALTER COLUMN base_salary COMMENT 'Current annual base salary in dollars.';
ALTER TABLE compensation
  ALTER COLUMN market_benchmark COMMENT 'Market benchmark salary in dollars for this role and level. Used to compare against base_salary.';
ALTER TABLE compensation
  ALTER COLUMN comp_ratio COMMENT 'Compensation ratio = base_salary / market_benchmark. A comp_ratio below 1.0 means the employee is paid below market rate (underpaid). A comp_ratio above 1.0 means paid above market rate. Use this column to identify underpaid employees.';
ALTER TABLE compensation
  ALTER COLUMN last_raise_date COMMENT 'Date of the employee most recent salary raise.';
ALTER TABLE compensation
  ALTER COLUMN last_raise_pct COMMENT 'Percentage of the most recent salary raise (e.g., 3.5 means a 3.5% raise).';

ALTER TABLE compensation ADD CONSTRAINT fk_compensation_employee FOREIGN KEY (employee_id) REFERENCES employees(employee_id) NOT ENFORCED;

-- -----------------------------------------------------------
-- resignations
-- -----------------------------------------------------------
COMMENT ON TABLE resignations
  IS 'Resignation records for employees who voluntarily left the company. Each row represents one resignation event. Use employee_id to join with employees to get employee details, and then employees.manager_id to join with managers for attrition analysis. Only employees with status = Resigned appear here.';

ALTER TABLE resignations
  ALTER COLUMN resignation_id COMMENT 'Unique identifier for each resignation record (e.g., RES-0001).';
ALTER TABLE resignations
  ALTER COLUMN employee_id COMMENT 'Foreign key to employees.employee_id. Identifies who resigned.';
ALTER TABLE resignations
  ALTER COLUMN resignation_date COMMENT 'Date the employee submitted their resignation.';
ALTER TABLE resignations
  ALTER COLUMN last_day COMMENT 'Employee final working day.';
ALTER TABLE resignations
  ALTER COLUMN reason_code COMMENT 'Primary reason for resignation. Valid values: Compensation, Career Growth, Manager Relationship, Personal, Relocation, Work-Life Balance.';
ALTER TABLE resignations
  ALTER COLUMN exit_survey_score COMMENT 'Exit survey satisfaction score on a 1.0 to 5.0 scale. Lower scores indicate more dissatisfaction.';

ALTER TABLE resignations ALTER COLUMN resignation_id SET NOT NULL;
ALTER TABLE resignations ADD CONSTRAINT pk_resignations PRIMARY KEY (resignation_id) NOT ENFORCED;
ALTER TABLE resignations ADD CONSTRAINT fk_resignations_employee FOREIGN KEY (employee_id) REFERENCES employees(employee_id) NOT ENFORCED;

-- Add FK from employees to managers (after managers PK exists)
ALTER TABLE employees ADD CONSTRAINT fk_employees_manager FOREIGN KEY (manager_id) REFERENCES managers(manager_id) NOT ENFORCED;

### C2. Verify Descriptions

1. Run the cell below to confirm descriptions and constraints were added.

In [0]:
DESCRIBE TABLE EXTENDED employees;

2. Open your **HR Genie Space** in a new tab. Select **Configure** > **Data** and confirm that:
   - Table descriptions appear at the top of each table
   - Column descriptions are visible for each column

   **NOTE:** You may need to refresh your Genie page if it was already open.

## D. Step 2: Add Column Synonyms and Prompt Matching

Beyond descriptions, the Knowledge Store supports **synonyms** and **prompt matching**.

Both of these are configured in the Genie Space UI.

### D1. Add Synonyms to Key Columns

Synonyms map **business language** to column names. 

Think about how your stakeholders talk about the data. They might say "salary" when they mean `base_salary`, or "pay" when they mean `compensation`.

In your HR Genie Space, add synonyms to the following columns. For each one:

1. Select **Configure** > **Data** > select the table.

2. Click the **pencil icon** next to the column.

3. Add a **Display Name** and **Synonyms**, then click **Save**.

| Table | Column | Display Name | Synonyms |
|-------|--------|-------------|----------|
| `compensation` | `base_salary` | `Base Salary` | `salary, pay, annual salary, current salary` |
| `compensation` | `comp_ratio` | `Comp Ratio` | `compensation ratio, pay ratio, market ratio, underpaid ratio` |
| `resignations` | `reason_code` | `Resignation Reason` | `reason, exit reason, departure reason, why they left` |
| `resignations` | `exit_survey_score` | `Exit Survey Score` | `exit score, satisfaction score, departure score` |

**NOTE:** Feel free to add additional synonyms to other columns based on how you think stakeholders would phrase their questions.

### D2. Enable Prompt Matching

For key categorical columns, confirm that **format assistance** is enabled and enable **entity matching** where appropriate.

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Prompt Matching Recap
  </strong>
  <div style="color:#333;">

- **Format assistance** — samples representative values so Genie understands data formats. Enabled by default on most columns.
- **Entity matching** — builds a complete list of distinct values for categorical columns. Use selectively on high-signal columns users frequently reference.

  </div>
</div>

Good candidates for entity matching in the HR dataset:

| Table | Column | Why |
|-------|--------|-----|
| `employees` | `status` | Users filter by Active, Resigned, Terminated |
| `employees` | `department` | Users ask about specific departments |
| `resignations` | `reason_code` | Users ask about resignation reasons by name |

For each column above:
1. Select **Configure** > **Data** > select the table > click the **pencil icon** next to the column.

2. Click **Advanced**.

3. Confirm **Format assistance** is **on**.

4. Confirm **Entity matching** is **on**.

5. Click **Save**.

**NOTE:** By default Genie will automatically set these for you so yours might already be enabled.

## E. Run Benchmarks After Adding Metadata

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Test After Every Phase
  </strong>
  <div style="color:#333;">

Each time you complete a phase of changes to your Genie Space, re-run benchmarks to confirm:
- Did the change **fix** the questions it was intended to fix?
- Did the change **break** anything that was already working?

  </div>
</div>

1. In your HR Genie Space, select **Benchmarks** > **Run all**.

2. Review the results. Did any previously failing benchmarks improve?

3. Take note of which benchmarks still need work.
    - These will be addressed by **SQL Logic** and **Instructions** in the next phases.

## F. Step 3: Add SQL Logic to the Knowledge Store

SQL logic addresses the benchmark failures that metadata alone could not fix. In this section, you will add:

| What | Why |
|------|-----|
| **Joins** | Define table relationships explicitly so Genie connects tables correctly |
| **SQL Expressions** | Create derived columns that do not exist in the data but can be computed |
| **Example SQL Queries** | Teach Genie complex query patterns it is unlikely to generate on its own |

### F1. Add a Join

While the foreign key constraints you added in the metadata phase give Genie join hints, explicitly defining joins in the Knowledge Store provides additional guidance. Especially for multi-table patterns.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Benchmark This Helps
  </strong>
  <div style="color:#333;">

**Benchmark #4: Which managers have the highest attrition on their teams?** — This requires joining `resignations > employees > managers`. Defining the employees-to-managers join explicitly helps Genie find this path.

  </div>
</div>

1. In your HR Genie Space, select **Configure** > **Instructions** > **Joins**.

2. Select **+ Add**.

3. Configure the join:
   - **Left Table:** `employees`
   - **Right Table:** `managers`
   - **Join Condition:** `manager_id` on both tables
   - **Relationship Type:** `Many to One`

4. Add the following to the **Instructions**: `Each employee has exactly one manager. Use this join when answering questions about manager performance, team composition, or attrition by manager. To calculate attrition by manager, join resignations to employees first (on employee_id), then employees to managers (on manager_id).`

5. Select **Save**.

### F2. Add a SQL Expression

SQL expressions let you define **derived columns**. 

These are values that do not exist in the data but can be computed from existing columns. Genie can use these expressions in generated SQL.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Benchmark This Helps
  </strong>
  <div style="color:#333;">

**Benchmark #3: Who are our most underpaid employees?** — Our business categorizes employees by compensation status based on their `comp_ratio`. This expression creates a `compensation_status` field that maps comp_ratio values to business categories.

  </div>
</div>

1. In your HR Genie Space, select **Configure** > **Instructions** > **SQL Expressions**.

2. Select **+ Add** > **Field**.

3. **Name the Field:** `compensation_status`

4. Run the cell below to generate the SQL **Code** with your catalog name.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
CASE
    WHEN " || current_catalog() || ".genie_course_hr.compensation.comp_ratio < 0.9
      THEN 'Significantly Underpaid'
    WHEN " || current_catalog() || ".genie_course_hr.compensation.comp_ratio < 1.0
      THEN 'Slightly Underpaid'
    WHEN " || current_catalog() || ".genie_course_hr.compensation.comp_ratio <= 1.1
      THEN 'At Market'
    ELSE 'Above Market'
END
";

SELECT my_query;

5. Copy the output above and paste into the **Code** field.

6. Add the following:
   - **Synonyms:** `pay status, comp status, salary status, underpaid status`
   - **Instructions:** `When users ask about compensation categories, pay bands, or underpaid status, use this expression. Employees with comp_ratio below 1.0 are considered underpaid. SCOPE_TABLES: Use with compensation and employees tables.`

7. Select **Save**.

### F3. Add Example SQL Queries

Example SQL queries teach Genie **query patterns** that it is unlikely to generate on its own. Genie uses these as trusted templates when similar questions are asked.

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Think of Example Queries as Teaching Patterns
  </strong>
  <div style="color:#333;">

One good example query can improve answers for many similar questions. Genie uses them in two ways:
- It can **use the query directly** when a question matches
- Or **learn from it** to generate similar queries for related questions

  </div>
</div>

#### F3a. Add Example Query: Who are our most underpaid employees?

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Benchmark This Fixes
  </strong>
  <div style="color:#333;">

**Benchmark #3: Who are our most underpaid employees?** — Genie does not know that "underpaid" means `comp_ratio < 1.0`. This query teaches the exact pattern.

  </div>
</div>

1. In your HR Genie Space, select **Configure** > **Instructions** > **SQL Queries**.

2. Select **+ Add** > **Example Query**.

3. Enter the question: `Who are our most underpaid employees?`

4. Run the cell below to generate the query with your catalog name, then paste it as the SQL query.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
SELECT
    e.employee_id,
    e.first_name,
    e.last_name,
    e.department,
    e.job_title,
    c.base_salary,
    c.market_benchmark,
    c.comp_ratio
FROM " || current_catalog() || ".genie_course_hr.employees e
JOIN " || current_catalog() || ".genie_course_hr.compensation c
    ON e.employee_id = c.employee_id
WHERE c.comp_ratio < 1.0
ORDER BY c.comp_ratio ASC
LIMIT 10
";

SELECT my_query;

5. Copy the output above and paste into the **SQL query** field.

6. Select **Preview** to verify the results look correct.

7. Expand **Usage Guidance** and add: `When users ask about underpaid employees, use comp_ratio < 1.0 as the filter. Return the most underpaid first (lowest comp_ratio). Default to top 10 unless specified otherwise.`

8. Select **Save**.

#### F3b. Add Example Query: Which managers have the highest attrition?

This teaches Genie the **three-table join** pattern needed to connect resignations to managers through employees.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Benchmark This Fixes
  </strong>
  <div style="color:#333;">

**Benchmark #4: Which managers have the highest attrition on their teams?** — This requires joining `resignations > employees > managers`. Genie is unlikely to discover this three-table join on its own.

  </div>
</div>

1. In your HR Genie Space, select **Configure** > **Instructions** > **SQL Queries**.

2. Select **+ Add** > **Example Query**.

3. Enter the question: `Which managers have the highest attrition on their teams?`

4. Run the cell below to generate the query with your catalog name, then paste it as the SQL query.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
SELECT
    m.manager_name,
    m.department,
    m.region,
    COUNT(r.resignation_id) AS attrition_count
FROM " || current_catalog() || ".genie_course_hr.resignations r
JOIN " || current_catalog() || ".genie_course_hr.employees e
    ON r.employee_id = e.employee_id
JOIN " || current_catalog() || ".genie_course_hr.managers m
    ON e.manager_id = m.manager_id
GROUP BY
    m.manager_name,
    m.department,
    m.region
ORDER BY attrition_count DESC
LIMIT 10
";

SELECT my_query;

5. Copy the output above and paste into the **SQL query** field.

6. Select **Preview** to verify the results look correct.

7. Expand **Usage Guidance** and add: `Attrition means the count of resignations per manager. Resignations do not link directly to managers — always join through the employees table. Return top 10 by default.`

8. Select **Save**.

## G. Run Benchmarks After Adding SQL Logic

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Checkpoint: Phase 2 Complete
  </strong>
  <div style="color:#333;">

You have now added a join, a SQL expression, and two example SQL queries. Re-run benchmarks to see the impact:
- Did the "underpaid" and "attrition" benchmarks improve?
- Did anything that was already passing break?

  </div>
</div>

1. In your HR Genie Space, select **Benchmarks** > **Run all**.

2. Review results. The "underpaid" and "attrition" benchmarks should show significant improvement.

3. If benchmarks are still failing, consider:
   - Does the SQL expression need adjustments?
   - Are the example queries using fully qualified table names?
   - You can also try **Knowledge Mining**: 
      - In **Benchmarks** > **Evaluations**
      - select your latest evaluation
      - select a failed benchmark
      - click **Review proposed fixes**.

## H. Step 4: Add Text Instructions - Final Touch

Text instructions define **business terms** and **default behaviors** that cannot be expressed in SQL or metadata alone. 

Use them for what metadata and SQL logic cannot handle.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    What Still Needs Defining?
  </strong>
  <div style="color:#333;">

Even with SQL logic in place, Genie may benefit from explicit behavioral guidance:
- **"Underpaid"** — Reinforce the definition: `comp_ratio < 1.0`
- **"Attrition"** — Reinforce the join path: `resignations → employees → managers`
- **Formatting** — Should currency be rounded? How many results by default?
- **Counting** — Should Genie use `COUNT(*)` or `COUNT(DISTINCT ...)`?

  </div>
</div>

### H1. Add Text Instructions

1. In your HR Genie Space, select **Configure** > **Instructions** > **Text**.

2. Add text instructions that reinforce your SQL logic and address general behaviors. Consider including:
   - A definition for **"underpaid"** (hint: `comp_ratio < 1.0`)
   - A definition for **"attrition"** (hint: count of resignations per manager, joined through employees)
   - Formatting rules (e.g., round salary values to 2 decimal places)
   - Default row limits (e.g., return top 10 unless specified otherwise)
   - Counting behavior (e.g., use `COUNT(*)` unless the user asks for distinct values)

3. Select **Save**.

**NOTE:** View the suggestion solution for ideas, or try your own.



##### SUGGESTED SOLUTIONS — TEXT INSTRUCTIONS

<details>
<summary>Click to expand suggested text instructions</summary>

<br/>

**NOTE:** These are suggestions. Your instructions may differ and still be effective. The key is that your instructions give Genie enough context to answer the benchmark questions correctly.

---

**Suggested Text Instructions**

```
- Underpaid Definition
    - "Underpaid" employees are those with a comp_ratio below 1.0
    - This means their base_salary is less than the market_benchmark
    - When asked about underpaid employees, filter on comp_ratio < 1.0 and order by comp_ratio ASC
    - Return the top 10 most underpaid employees by default unless specified otherwise
    - Include: employee_id, first_name, last_name, department, job_title, base_salary, market_benchmark, comp_ratio

- Attrition Definition
    - "Attrition" means the count of resignations
    - To find attrition by manager, join resignations to employees (on employee_id), then employees to managers (on manager_id)
    - Resignations do not link directly to managers — always join through the employees table
    - When asked about manager attrition, return manager_name, department, region, and resignation count
    - Order by resignation count descending, return top 10 by default

- Formatting
    - Round all salary and currency values to 2 decimal places
    - Format using US dollars

- General Rules
    - When counting rows, use COUNT(*) unless the user asks for distinct values
    - Return top 10 results by default when ranking or listing "top" items
```

</details>


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Best Practice
  </strong>
  <div style="color:#333;">

Keep text instructions **short and specific**. Avoid duplicating what is already defined in column descriptions, SQL expressions, or example queries. Instructions are for behavior and edge cases only.

  </div>
</div>

## I. Run Final Benchmarks

1. In your HR Genie Space, select **Benchmarks** > **Run all**.

2. Review your final results. The goal is that **all benchmarks pass**.

3. If some benchmarks still fail, consider:
   - Are your instructions specific enough?
   - Try **Knowledge Mining**: In **Benchmarks** > **Evaluations**, select your latest evaluation > select a failed benchmark > click **Review proposed fixes**. Review the suggested knowledge snippets and accept any that look correct.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Results May Vary
  </strong>
  <div style="color:#333;">

Even with all three phases complete, Genie may not match your ground truth SQL exactly. 

The goal is **directional improvement**.

Are the results closer to what you expect? Over time, you refine further by adding more examples, adjusting instructions, and reviewing user feedback through Monitoring.

  </div>
</div>


<div style="
  border-left: 4px solid #7b1fa2;
  background: #f3e5f5;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#4a148c; margin-bottom:6px; font-size: 1.1em;">Continue Practicing</strong>
  <div style="color:#333;">

This was a simple starter Genie Space. There is much more you can do to improve accuracy and consistency. 

Continue building by adding additional benchmarks, refining SQL logic, enhancing metadata, and expanding your test coverage.

Treat your Genie Space like a product. Iterate, test, and improve over time.

  </div>
</div>

## J. BONUS - Agent Mode in Genie Spaces
Agent mode is designed for deeper, multi-step investigation.

The questions should be open-ended and analytical rather than single-query answers.

- Agent mode in Genie spaces:
[AWS](https://docs.databricks.com/aws/en/genie/agent-mode) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/genie/agent-mode) |
[GCP](https://docs.databricks.com/gcp/en/genie/agent-mode)



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

Workspace admins can control access to Genie Agent mode (Public Preview) using the previews page. 

After Agent mode is enabled, it is available for all Genie spaces within that workspace. All users with access to a Genie space can use Agent mode. All requirements for accessing a Genie space and its data apply. See Set up a Genie space.


  </div>
</div>

1. Navigate to your Genie Space.

2. Select **Agent** (instead of **Chat**)

3. Ask the following questions (or anything else you can think of) and view the results:

1. `Investigate what's driving employee resignations. Are there patterns by department, manager, or compensation?`
   - Forces the agent to explore across all four tables, including resignation reasons, compensation ratios of resigned employees, manager performance ratings, and department-level breakdowns. This requires multi-step analysis rather than a single query.

2. `Which departments have the biggest compensation equity problems, and who are the most at-risk employees?`
   - The agent would need to analyze compensation ratio distributions by department, identify underpaid clusters, cross-reference job levels and tenure, and surface specific employees. This requires iterative exploration across multiple dimensions.

3. `Compare attrition patterns between high-performing and low-performing managers. What do you find?`
   - The agent needs to join manager performance ratings to employee and resignation data, segment managers into groups, and compare resignation counts, exit survey scores, and reason codes. This is a hypothesis-driven investigation, not a simple retrieval task.

## K. Conclusion

In this lab, you improved your HR Genie Space using the three-phase approach:

**Phase 1 — Metadata:**
- Added **table and column descriptions** to give Genie context about your HR data.
- Added **synonyms** to bridge the gap between stakeholder language and column names.
- Enabled **prompt matching** on key categorical columns.

**Phase 2 — SQL Logic:**
- Added a **join** to define the employees-to-managers relationship.
- Added a **SQL expression** to create a `compensation_status` field.
- Added **example SQL queries** to teach Genie complex patterns (underpaid employees and manager attrition).

**Phase 3 — Instructions:**
- Added **text instructions** to define ambiguous business terms like "underpaid" and "attrition" and set default behaviors.

You **re-ran benchmarks** after each phase to measure improvement and catch regressions.

You have now completed the full Genie curation lifecycle: from raw tables to a fully curated, benchmarked Genie Space.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>